# 5주차 2번 과제: 한국어 챗봇


In [11]:
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import PreTrainedTokenizerFast

random.seed(42)
torch.manual_seed(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    print('MPS 없음 -> CPU 사용')

print('device:', device)

device: mps


In [12]:
DATA_PATH = Path('Dataset/kanana_persona_sft_v1_final_50k.jsonl')
SAVE_PATH = Path('checkpoints/chatbot.pt')
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

TOKENIZER_NAME = 'kakaocorp/kanana-nano-2.1b-instruct'
MAX_LEN = 256
TRAIN_SIZE = 1000
VAL_SIZE = 100
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 0.001
EARLY_STOP = 3

EMBED_SIZE = 256
N_HEADS = 4
NUM_LAYERS = 2
FFN_SIZE = 512

In [13]:
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_id = tokenizer.pad_token_id
eos_id = tokenizer.eos_token_id
vocab_size = len(tokenizer)

print('단어 개수(vocab):', vocab_size)

단어 개수(vocab): 128256


In [14]:
train_data = []
val_data = []

with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('split') == 'validation':
            val_data.append(row)
        else:
            train_data.append(row)

random.shuffle(train_data)
random.shuffle(val_data)
train_data = train_data[:TRAIN_SIZE]
val_data = val_data[:VAL_SIZE]

print('학습 데이터:', len(train_data))
print('검증 데이터:', len(val_data))

학습 데이터: 1000
검증 데이터: 100


In [15]:
def make_input_label(messages):
    ids = []
    labels = []

    for msg in messages:
        role = msg['role']
        text = f"<|{role}|>\n{msg['content'].strip()}\n"
        token_ids = tokenizer.encode(text, add_special_tokens=False)

        ids.extend(token_ids)
        if role == 'assistant':
            labels.extend(token_ids)
        else:
            labels.extend([-100] * len(token_ids))

    if len(ids) > MAX_LEN:
        ids = ids[-MAX_LEN:]
        labels = labels[-MAX_LEN:]

    return torch.tensor(ids), torch.tensor(labels)


class ChatDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return make_input_label(self.rows[idx]['messages'])


def collate_fn(batch):
    xs, ys = zip(*batch)
    x = pad_sequence(xs, batch_first=True, padding_value=pad_id)
    y = pad_sequence(ys, batch_first=True, padding_value=-100)
    return x, y


train_loader = DataLoader(ChatDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ChatDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

sample_x, _ = make_input_label(train_data[0]['messages'])
print('샘플 길이:', len(sample_x))

샘플 길이: 256


In [16]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_size, n_heads, ffn_size):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_size, n_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(embed_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, embed_size),
        )
        self.ln1 = nn.LayerNorm(embed_size)
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x, attn_mask):
        out, _ = self.attn(x, x, x, attn_mask=attn_mask, need_weights=False)
        x = self.ln1(x + out)
        x = self.ln2(x + self.ff(x))
        return x


class ChatModel(nn.Module):
    def __init__(self, vocab_size, embed_size, n_heads, num_layers, ffn_size, max_len, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.max_len = max_len
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_id)
        self.pos_embedding = nn.Embedding(max_len, embed_size)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_size, n_heads, ffn_size)
            for _ in range(num_layers)
        ])
        self.ln = nn.LayerNorm(embed_size)
        self.fc = nn.Linear(embed_size, vocab_size)

    def _causal_mask(self, seq_len, device):
        return torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=device),
            diagonal=1,
        )

    def forward(self, x):
        seq_len = x.size(1)
        pos = torch.arange(seq_len, device=x.device)
        h = self.embedding(x) + self.pos_embedding(pos)

        attn_mask = self._causal_mask(seq_len, x.device)
        for layer in self.layers:
            h = layer(h, attn_mask)

        return self.fc(self.ln(h))


model = ChatModel(
    vocab_size, EMBED_SIZE, N_HEADS, NUM_LAYERS, FFN_SIZE, MAX_LEN, pad_id
).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

ChatModel(
  (embedding): Embedding(128256, 256, padding_idx=128001)
  (pos_embedding): Embedding(256, 256)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=512, out_features=256, bias=True)
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (fc): Linear(in_features=256, out_features=128256, bias=True)
)


In [17]:
def calc_loss(x, y):
    logits = model(x[:, :-1])
    target = y[:, 1:]
    return criterion(logits.reshape(-1, vocab_size), target.reshape(-1))


def calc_val_loss():
    model.eval()
    total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            total += calc_loss(x, y).item()
    return total / len(val_loader)


best_val_loss = float('inf')
bad_count = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = calc_loss(x, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss = calc_val_loss()
    print(f'epoch {epoch + 1}/{EPOCHS} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        bad_count = 0
        torch.save(
            {
                'model': model.state_dict(),
                'config': {
                    'tokenizer': TOKENIZER_NAME,
                    'vocab_size': vocab_size,
                    'max_len': MAX_LEN,
                    'embed_size': EMBED_SIZE,
                    'n_heads': N_HEADS,
                    'num_layers': NUM_LAYERS,
                    'ffn_size': FFN_SIZE,
                    'pad_id': pad_id,
                },
            },
            SAVE_PATH,
        )
        print('  -> 모델 저장:', SAVE_PATH)
    else:
        bad_count += 1
        if bad_count >= EARLY_STOP:
            print(f'  -> val loss가 {EARLY_STOP}번 연속 개선되지 않아 학습 종료')
            break

print('학습 끝. best val loss:', round(best_val_loss, 4))

epoch 1/5 | train loss: 5.2982 | val loss: 2.5436
  -> 모델 저장: checkpoints/chatbot.pt
epoch 2/5 | train loss: 2.1831 | val loss: 1.4865
  -> 모델 저장: checkpoints/chatbot.pt
epoch 3/5 | train loss: 1.4026 | val loss: 1.0725
  -> 모델 저장: checkpoints/chatbot.pt
epoch 4/5 | train loss: 0.9965 | val loss: 0.8854
  -> 모델 저장: checkpoints/chatbot.pt
epoch 5/5 | train loss: 0.7512 | val loss: 0.7763
  -> 모델 저장: checkpoints/chatbot.pt
학습 끝. best val loss: 0.7763


In [18]:
def predict_next_word(prompt):
    model.eval()
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        next_id = model(x)[0, -1].argmax().item()

    return tokenizer.decode([next_id])


test_prompt = (
    '<|system|>\n너는 친근한 한국어 챗봇이다.\n'
    '<|user|>\n오늘 기분이 별로야.\n'
    '<|assistant|>\n'
)

next_word = predict_next_word(test_prompt)
print('입력:', test_prompt)
print('다음 단어:', repr(next_word))

입력: <|system|>
너는 친근한 한국어 챗봇이다.
<|user|>
오늘 기분이 별로야.
<|assistant|>

다음 단어: '그'


In [19]:
def make_prompt(system_text, user_text):
    return (
        f'<|system|>\n{system_text.strip()}\n'
        f'<|user|>\n{user_text.strip()}\n'
        f'<|assistant|>\n'
    )


def chat(system_text, user_text, max_words=80):
    model.eval()
    prompt = make_prompt(system_text, user_text)
    ids = tokenizer.encode(prompt, add_special_tokens=False)

    for _ in range(max_words):
        x = torch.tensor([ids[-MAX_LEN:]], dtype=torch.long, device=device)
        with torch.no_grad():
            next_id = model(x)[0, -1].argmax().item()
        ids.append(next_id)
        if next_id == eos_id:
            break

    text = tokenizer.decode(ids)
    return text.split('<|assistant|>')[-1].strip()


system = '너는 ENFP 성향의 20대 후반 성인 여성 친구다.'
user = '오늘 괜히 답장 하나 때문에 기분이 흔들렸어.'

answer = chat(system, user)
print('user:', user)
print('bot :', answer)

user: 오늘 괜히 답장 하나 때문에 기분이 흔들렸어.
bot : 그건 네 설정에는 긴을 정리감 먼저야.
<|>
그럼 옷, 괜찮아. 오늘은 큰 결로.
<|>
그럼 새벽지는 말고 흔들려
